# LegalLens RAG Evaluation

Three pipelines, run in order:

1. **Retrieval evaluation (fast, offline)** — measures whether the right passages
   reach the model, using a 200-question gold set (`gold_set.json`, built by
   `build_gold_set.py` from verified corpus sections).
   Metrics: Hit@k, Precision@k, Recall@k, MRR, nDCG@8. Every retrieval config is
   compared in an ablation table:
   `faiss only` → `+ BM25/RRF` → `+ citation anchoring` → `web fallback`.
2. **Answer evaluation (optional, needs internet)** — runs the full agent on Groq
   (this device has no Ollama) and checks required content markers in each answer.
3. **Conversation replay** — re-runs the real 5-turn Section-379 transcript that failed
   before the fixes, and verifies the final turns now quote the provision.

> No server needed — the notebook calls the pipeline functions directly.
> The first import loads the FAISS indexes and embedding model (a few minutes, once).

**Honesty rule for the report:** only quote numbers produced by actually running these
cells, and state the gold-set size (203), Act coverage (56) and query types.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\Main\Desktop\Lex-AI")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# The machine-wide NO_PROXY may contain "[::1]", which crashes the httpx/ollama
# imports; this kernel only ever needs localhost.
os.environ["NO_PROXY"] = "localhost,127.0.0.1"

import json
import math
import time

import legal_agent as la
import hybrid_retrieval
import pakistancode_web as pw

print(json.dumps(la.retrieval_status(), indent=2)[:800])

{
  "law": {
    "state": "pending",
    "path": "C:\\Users\\Main\\Desktop\\Lex-AI\\faiss_index_law",
    "vectors": 0,
    "error": null,
    "last_query_error": null
  },
  "cases": {
    "state": "pending",
    "path": "C:\\Users\\Main\\Desktop\\Lex-AI\\faiss_index_cases",
    "vectors": 0,
    "error": null,
    "last_query_error": null
  }
}



## 1. Gold standard (~200 questions)

`gold_set.json` is built by `build_gold_set.py` (fixed random seed → reproducible).
Every entry is grounded in the real corpus: a numbered section whose full text is
indexed, labelled with the Act + **all page numbers** containing it (so duplicate
Act copies count as correct). Recall is computed over these distinct answer pages;
`n_relevant` (indexed chunk count) is kept for auditing. Query styles:

- **citation** — names the section number and the Act ("what is section 469 of the
  Companies Ordinance");
- **natural** — asks by topic ("what is punishment for theft"); when the heading
  is unique across the corpus the Act is not named, otherwise it must be, so no
  gold entry is unfairly ambiguous.

To rebuild: `python build_gold_set.py`. To add hand-labelled entries (e.g. the
web-only `out_of_corpus` case), verify pages with `find_gold()` below and append
the same dict shape to `GOLD`.

In [2]:
GOLD_PATH = Path("evaluations/gold_set.json")
if GOLD_PATH.exists():
    GOLD = json.loads(GOLD_PATH.read_text(encoding="utf-8"))
else:
    GOLD = []
    print("gold_set.json missing -> run:  python evaluations/build_gold_set.py")

# Verified web-only law: absent from the local corpus, present on
# pakistancode.gov.pk — the correct system behaviour is to serve it from
# the web fallback, so it is judged only against the "web fallback" config.
GOLD.append({"query": "what does the Akhuwat Institute Kasur Act 2026 provide",
             "act": "Akhuwat", "pages": "any", "type": "out_of_corpus",
             "expected_source": "web"})

acts = {e["act"] for e in GOLD}
types = {t: sum(1 for e in GOLD if e["type"] == t) for t in {e["type"] for e in GOLD}}
print(f"loaded {len(GOLD)} gold queries | {len(acts)} Acts | {types}")

Loading law FAISS index from 'C:\Users\Main\Desktop\Lex-AI\faiss_index_law'...
Loading embedding model 'BAAI/bge-base-en-v1.5' on 'cpu'...
C:\Users\Main\Desktop\Lex-AI\legal_agent.py:183: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  _embeddings = HuggingFaceEmbeddings(
loaded 204 gold queries | 57 Acts | {'citation': 135, 'out_of_corpus': 1, 'natural': 68}



In [3]:
def find_gold(act_kw, phrase, kind="law", limit=12):
    """List index chunks of one Act containing the exact answer text.
    Use the printed pages to fill new GOLD entries."""
    store = la.get_store(kind)
    rows = set()
    for doc in store.docstore._dict.values():
        m = doc.metadata or {}
        src = str(m.get("source", "")) + " " + str(m.get("act_name", ""))
        if act_kw.lower() in src.lower() and phrase.lower() in doc.page_content:
            name = src.replace("\\", "/").split("/")[-1][:60]
            rows.add((str(m.get("page_label")), name))
    for page, name in sorted(rows)[:limit]:
        print(f"page {page:>4} | {name}")
    return sorted({int(p) for p, _ in rows if p.isdigit()})

# example: find_gold("penal code", "Punishment for theft. Whoever commits") -> [137]

## 2. Metrics — what each one says

The retriever returns a ranked list; a hit requires **Act family and page both**
matching the gold label. `k = 8` is roughly the evidence budget shown to the model.

| metric | plain meaning | best | how to read our numbers |
|---|---|---|---|
| **Hit@k** | did *at least one* correct chunk make it into the top k? ("did retrieval work at all") | 1.0 | Hit@1 = 0.60 means 60% of questions have the answer in the very first slot |
| **Precision@k** | what share of the top k is correct? ("was the context clean or noisy?") | 1.0 | equals Hit@1 when one chunk holds the answer |
| **Recall@k** | what share of the labelled **answer pages** appear in the top k? ("did we recover every place the answer lives, incl. duplicate Act copies?") | 1.0 | below Hit when only some of several answer pages are retrieved |
| **MRR** | 1 ÷ rank of the first correct chunk ("how high did the right passage climb?") | 1.0 | 0.25 ≈ first correct hit around rank 4; chunks at rank 8 are invisible once evidence budget is 3–5 |
| **nDCG@8** | one rank-quality score: correct chunks early = high, late = discounted | 1.0 | use for overall ordering quality between configs |

**In one line for the defence:** Hit answers "did it work", Recall "did it get
everything", MRR/nDCG "how well was it ranked". A working RAG needs high Hit@5,
acceptable Recall@8, and MRR close to the top because the generator only reads
the first few sources.

In [4]:
K_EVAL = [1, 3, 5, 8]

def normalize_act(name):
    return la._act_family({"act_name": name})

def hit_pages(hit, gold):
    """The gold pages this hit lands on; empty set when it is not a match."""
    m = hit["doc"].metadata or {}
    fam = la._act_family(m)
    act_slug = normalize_act(gold["act"])
    if not (fam.startswith(act_slug) or act_slug.startswith(fam)):
        return set()
    if gold["pages"] == "any":
        return {"any"}
    page = str(m.get("page_label") or m.get("page") or m.get("page_number") or "")
    return {page} if page in {str(p) for p in gold["pages"]} else set()

def dcg(rel):
    return sum(r / math.log2(i + 2) for i, r in enumerate(rel))

def score_query(retriever, gold, top_k=8):
    hits = retriever(gold["query"], top_k)
    rel = [bool(hit_pages(h, gold)) for h in hits[:top_k]]
    rank = next((i + 1 for i, v in enumerate(rel) if v), None)
    # Recall over distinct ANSWER PAGES: several hits can share one page, so
    # counting raw hits against a chunk-based denominator would exceed 1.0.
    total_pages = len(gold["pages"]) if gold["pages"] != "any" else 1
    seen = set()
    row = {"query": gold["query"], "type": gold["type"], "rank": rank}
    for k in K_EVAL:
        row[f"hit@{k}"] = float(any(rel[:k]))
        row[f"precision@{k}"] = sum(rel[:k]) / k
        for h in hits[:k]:
            seen |= hit_pages(h, gold)
        row[f"recall@{k}"] = min(1.0, len(seen) / total_pages)
    idcg = dcg([1.0] * sum(rel))
    row["ndcg@8"] = dcg(rel) / idcg if idcg else 0.0
    row["mrr"] = 1.0 / rank if rank else 0.0
    return row

## 3. Retrieval configs (the ablation)

Each column isolates one engineering contribution:

| config | what it is |
|---|---|
| `faiss only` | dense semantic search as it was before any fixes |
| `faiss + bm25 (rrf)` | hybrid search with weighted Reciprocal Rank Fusion |
| `faiss + bm25 + citation` | + citation anchoring (guarantees "379." chunks) |
| `web fallback` | official Pakistan Code texts for laws missing locally (network, slow first run) |

In [5]:
def retrieve_faiss_only(query, top_k):
    hybrid_retrieval.HYBRID_ENABLED = False
    try:
        return la._search_store("law", query, top_k)
    finally:
        hybrid_retrieval.HYBRID_ENABLED = True

def retrieve_hybrid(query, top_k):
    return la._search_store("law", query, top_k)

def retrieve_hybrid_citation(query, top_k):
    return la._citation_boost("law", query, la._search_store("law", query, top_k))

def retrieve_web_fallback(query, top_k):
    try:
        return pw.search_web_laws([query])
    except Exception as exc:
        print("web error:", type(exc).__name__, exc)
        return []

RETRIEVERS = {
    "faiss only": retrieve_faiss_only,
    "faiss + bm25 (rrf)": retrieve_hybrid,
    "faiss + bm25 + citation": retrieve_hybrid_citation,
    "web fallback": retrieve_web_fallback,
}

In [6]:
RUN_WEB = True   # also evaluates the Pakistan Code fallback (needs internet; first run builds the catalog cache)

rows = []
configs = [c for c in RETRIEVERS if RUN_WEB or c != "web fallback"]
for name in configs:
    print(f"\n--- {name} ---")
    for gold in GOLD:
        is_web = gold.get("expected_source") == "web"
        if is_web != (name == "web fallback"):
            continue  # site-only laws are only judged against the web config
        t0 = time.time()
        try:
            r = score_query(RETRIEVERS[name], gold)
        except Exception as exc:
            r = {"query": gold["query"], "type": gold["type"], "error": str(exc)}
        rows.append({"config": name, "seconds": round(time.time() - t0, 2), **r})
        print(f"  hit@8={r.get('hit@8', '-')} rank={r.get('rank', '-')} "
              f"{round(time.time()-t0, 2)}s | {r['query'][:58]}")


--- faiss only ---
Embedding model ready.
Law FAISS index ready (51,475 vectors).
  hit@8=0.0 rank=None 14.45s | what is section 10 of the Virtual University Ordinance
  hit@8=0.0 rank=None 0.27s | tell me about section 10 (Virtual University Ordinance)
  hit@8=1.0 rank=1 0.55s | what is finance and planning council in pakistani law
  hit@8=0.0 rank=None 0.28s | what is section 7 of the Associated Press of Pakistan Ordi
  hit@8=0.0 rank=None 0.26s | tell me about section 7 (Associated Press of Pakistan Ordi
  hit@8=1.0 rank=2 0.42s | what is officers and employees of the associated press in 
  hit@8=1.0 rank=5 0.25s | what is section 8 of the Naval Armaments Act
  hit@8=1.0 rank=5 0.31s | tell me about section 8 (Naval Armaments Act)
  hit@8=1.0 rank=1 0.29s | what is procedure in forfeiture of ships in pakistani law
  hit@8=0.0 rank=None 0.35s | what is section 9 of the National Insurance Corporation Ac
  hit@8=0.0 rank=None 0.31s | tell me about section 9 (National Insurance Corpora

In [7]:
def mean(vals):
    return sum(vals) / len(vals) if vals else 0.0

print(f"{'config':26} {'hit@1':>6} {'hit@3':>6} {'hit@5':>6} {'hit@8':>6} "
      f"{'rec@8':>6} {'mrr':>6} {'ndcg@8':>7}")
for name in configs:
    sub = [r for r in rows if r["config"] == name and not r.get("error")]
    if not sub:
        continue
    cols = {k: mean([r[f"hit@{k}"] for r in sub]) for k in K_EVAL}
    print(f"{name:26} {cols[1]:6.2f} {cols[3]:6.2f} {cols[5]:6.2f} {cols[8]:6.2f} "
          f"{mean([r['recall@8'] for r in sub]):6.2f} "
          f"{mean([r['mrr'] for r in sub]):6.3f} "
          f"{mean([r['ndcg@8'] for r in sub]):7.3f}")
    for t in sorted({r["type"] for r in sub}):
        ss = [r for r in sub if r["type"] == t]
        print(f"    {t:14} n={len(ss):3}  hit@8 {mean([r['hit@8'] for r in ss]):5.2f}"
              f"  rec@8 {mean([r['recall@8'] for r in ss]):5.2f}"
              f"  mrr {mean([r['mrr'] for r in ss]):5.3f}")

Path("evaluations/eval_results.json").write_text(
    json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
print("\nsaved -> evaluations/eval_results.json")

config                      hit@1  hit@3  hit@5  hit@8  rec@8    mrr  ndcg@8
faiss only                   0.24   0.34   0.44   0.49   0.49  0.312   0.357
    citation       n=135  hit@8  0.34  rec@8  0.34  mrr 0.169
    natural        n= 68  hit@8  0.79  rec@8  0.79  mrr 0.596
faiss + bm25 (rrf)           0.23   0.33   0.38   0.49   0.49  0.306   0.350
    citation       n=135  hit@8  0.33  rec@8  0.33  mrr 0.136
    natural        n= 68  hit@8  0.82  rec@8  0.82  mrr 0.643
faiss + bm25 + citation      0.61   0.80   0.83   0.85   0.85  0.704   0.729
    citation       n=135  hit@8  0.86  rec@8  0.86  mrr 0.734
    natural        n= 68  hit@8  0.82  rec@8  0.82  mrr 0.643
web fallback                 1.00   1.00   1.00   1.00   1.00  1.000   1.000
    out_of_corpus  n=  1  hit@8  1.00  rec@8  1.00  mrr 1.000

saved -> eval_results.json


## 4. Answer evaluation (end-to-end agent)

Runs the **full pipeline** (context resolution → decomposition → hybrid search →
citation boost → fallback → generation on **Groq**, since this device has no Ollama)
and checks content markers. For a defence-grade number also score the saved answers
by hand on a 1–5 rubric (correctness / grounding / clarity); report both the marker
pass-rate and the human mean.

In [8]:
RUN_ANSWER_EVAL = True   # needs internet + GROQ_API_KEY from .env

# Markers below were verified against live Groq answers in the Section-379
# investigation (379 -> "three years", 378 -> definition of theft). Add more
# only after confirming the expected content from a real run.
ANSWER_GOLD = [
    {"query": "what is section 379 of pakistan penal code",
     "must_contain": ["379", "three years"]},
    {"query": "what is the punishment for theft under PPC",
     "must_contain": ["three years"]},
    {"query": "what is section 378 of pakistan penal code",
     "must_contain": ["378"]},
    {"query": "how is khula granted in family law",
     "must_contain": ["family court"]},
]

if RUN_ANSWER_EVAL:
    import uuid
    sid = "eval-" + uuid.uuid4().hex[:8]
    out = []
    for item in ANSWER_GOLD:
        t0 = time.time()
        r = la.ask_legal_agent(item["query"], session_id=sid, provider="groq")
        ans = r.get("answer") or ""
        low = ans.lower()
        ok = all(m.lower() in low for m in item["must_contain"])
        out.append({"query": item["query"], "answer": ans,
                    "sources": r.get("sources"), "markers": item["must_contain"],
                    "markers_ok": ok, "elapsed_s": round(time.time() - t0, 1)})
        print(f"[{'PASS' if ok else 'FAIL'}] {item['query'][:60]} "
              f"({out[-1]['elapsed_s']}s)")
    Path("evaluations/eval_answers.json").write_text(
        json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
    passed = sum(o["markers_ok"] for o in out)
    print(f"answer eval: {passed}/{len(out)} marker pass-rate "
          f"({passed/len(out):.0%})  saved -> evaluations/eval_answers.json")

Embedding model ready.
Law FAISS index ready (51,475 vectors).
BM25 law index loaded from cache (51,475 chunks).
Loading cases FAISS index from 'C:\Users\Main\Desktop\Lex-AI\faiss_index_cases'...
Cases FAISS index ready (209,168 vectors).
BM25 cases index loaded from cache (209,168 chunks).
[LLM] legal answer: input=3307 output=256 reasoning=27 total=3563
[PASS] what is section 379 of pakistan penal code (30.3s)
Pakistan Code web fallback supplied 4 passages.
[LLM] legal answer: input=3489 output=232 reasoning=72 total=3721
[FAIL] what is the punishment for theft under PPC (2.0s)
Pakistan Code web fallback supplied 2 passages.
[LLM] legal answer: input=3619 output=234 reasoning=31 total=3853
[PASS] what is section 378 of pakistan penal code (10.4s)
Pakistan Code web fallback supplied 4 passages.
[LLM] legal answer: input=3423 output=412 reasoning=36 total=3835
[PASS] how is khula granted in family law (28.1s)
answer eval: 3/4 marker pass-rate (75%)  saved -> eval_answers.json



## 5. Conversation replay (adversarial multi-turn)

The real 5-turn transcript from the Section-379 failure, replayed in one session.
Pass criteria per turn encode what the *correct* answer contains — this tests context
resolution, decomposition, dedupe, citation boosting and fallback together.

In [9]:
RUN_CONVERSATION_REPLAY = True   # needs internet + GROQ_API_KEY

CONVERSATION = [
    ("what section 379 of pakistan penal code says", ["379", "theft"]),
    ("what is theft", ["378"]),
    ("what is punishment for theft", ["three years"]),
    ("what is section 379 of pakistan penal code", ["379", "three years"]),
    ("PPC section 379", ["three years"]),
]

if RUN_CONVERSATION_REPLAY:
    import uuid
    sid = "eval-convo-" + uuid.uuid4().hex[:8]
    results = []
    for q, markers in CONVERSATION:
        r = la.ask_legal_agent(q, session_id=sid, provider="groq")
        ans = (r.get("answer") or "").lower()
        ok = all(m.lower() in ans for m in markers)
        results.append({"query": q, "markers": markers, "pass": ok,
                        "answer": r.get("answer"), "sources": r.get("sources")})
        print(f"[{'PASS' if ok else 'FAIL'}] {q}")
    passed = sum(x["pass"] for x in results)
    print(f"conversation: {passed}/{len(results)} turns pass "
          "(before the fixes this replay scored 0/5)")
    Path("evaluations/eval_conversation.json").write_text(
        json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved -> evaluations/eval_conversation.json")

[LLM] legal answer: input=3323 output=119 reasoning=24 total=3442
[PASS] what section 379 of pakistan penal code says
[LLM] follow-up resolution: input=221 output=70 reasoning=38 total=291
[LLM] question decomposition: input=284 output=260 reasoning=145 total=544
[LLM] legal answer: rate-limited, retrying in 8.0s
[LLM] legal answer: input=3556 output=257 reasoning=41 total=3813
[PASS] what is theft
[LLM] follow-up resolution: input=277 output=63 reasoning=40 total=340
[LLM] legal answer: input=3198 output=92 reasoning=26 total=3290
[PASS] what is punishment for theft
[LLM] legal answer: input=3619 output=86 reasoning=23 total=3705
[PASS] what is section 379 of pakistan penal code
[LLM] follow-up resolution: input=286 output=55 reasoning=35 total=341
Pakistan Code web fallback supplied 4 passages.
[LLM] legal answer: input=3317 output=212 reasoning=32 total=3529
[PASS] PPC section 379
conversation: 5/5 turns pass (before the fixes this replay scored 0/5)


## How to present the numbers

- The **ablation table** is the centrepiece: each column isolates one engineering
  contribution (hybrid BM25 fusion, citation anchoring, web fallback) and the
  per-query-type split shows *where* each one earns its cost.
- Report **per query type**, not only the average — averages hide the failure
  mode we fixed (citation questions).
- Quote **latency** from `eval_results.json` (`seconds` column) when discussing
  the quality/latency trade-off of the web fallback.

## Limitations to disclose (honest defence)

- The gold set is **auto-grounded in the corpus** (203 queries, 56 Acts, seed 42):
  pages were verified against the index, but queries follow generation templates
  and a single annotator labelled them. Real user phrasing may differ.
- Relevance is decided at **page granularity** (any chunk from a labelled answer
  page counts), the practical compromise for a PDF-derived corpus with duplicate
  Act copies; exact-chunk labelling would report slightly lower scores.
- Marker checks in the answer eval are proxies for correctness; human rubric
  scores are the primary generation metric.
- No LLM-as-judge was used — deliberate: a Groq judge scoring Groq answers
  shares biases.